# 3.2 權重能由內容決定嗎？

# 第 3 章：Attention，自己決定看哪裡

前置：向量、矩陣乘法和 softmax。先用單 head；每個小節都畫得出矩陣。完整模型仍留到第4章。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：把分數換成比例：像分一塊餅**

一起加同常數不改比例，分數差才重要。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/softmax.svg")))

**先想一想：**q=[1,0] 比較偏好哪個 key？

用點積看兩個向量是否朝相近方向，再把分數 softmax 成權重。現在 query 改變就會改變讀取偏好。

**把直覺寫成數學：**$s_j=q\cdot k_j$，$a=softmax(s)$。點積同時受方向與長度影響。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
q = torch.tensor([1.0, 0.0])
keys = torch.tensor([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]])
scores = q @ keys.T
print(scores, scores.softmax(0))

**如何讀結果：**這裡還沒有 causal 限制；不能直接用來訓練自回歸模型。

**只改一件事：**只把 q 改成 [0,1]。
